In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples

df = pd.read_csv("../data/processed/customers_clean.csv")
pipeline = joblib.load("../models/customeriq_kmeans_pipeline.joblib")

FEATURES = ["Number_of_Orders", "Avg_Order_Value", "Days_Since_Last_Purchase",
            "Items_Per_Order", "Average_Rating", "Discount_Applied"]

X_scaled = pipeline.named_steps["preprocess"].transform(df[FEATURES])
base_labels = pipeline.predict(df[FEATURES])
print(X_scaled.shape)

(60000, 6)


In [2]:
#how well-separated are the clusters?
rng = np.random.default_rng(42)
idx = rng.choice(len(X_scaled), 10000, replace=False)
sil = silhouette_samples(X_scaled[idx], base_labels[idx])

print("Overall silhouette:", sil.mean().round(3))
print()
print(pd.Series(sil).groupby(base_labels[idx]).mean().round(3))
print()
print("Share of customers with a negative score:", (sil < 0).mean().round(3))

Overall silhouette: 0.25

0    0.207
1    0.248
2    0.319
3    0.236
dtype: float64

Share of customers with a negative score: 0.011


In [ ]:
results = []
for seed in range(1, 11):
    km = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=seed).fit(X_scaled)
    results.append({
        "seed": seed,
        "inertia": round(km.inertia_),
        "ARI_vs_final": round(adjusted_rand_score(base_labels, km.labels_), 3),
    })
pd.DataFrame(results)
#This retrains K-Means 10 times, each with a different random seed, and compares each result with our saved model.

,seed,inertia,ARI_vs_final
0,1,175664,0.999
1,2,175665,1.000
2,3,175664,0.998
3,4,175665,1.000
4,5,175665,1.000
5,6,175664,0.999
6,7,175665,0.999
7,8,175664,0.998
8,9,175665,1.000
9,10,175665,1.000


In [4]:
single_runs = []
for seed in range(1, 11):
    km = KMeans(n_clusters=4, init="random", n_init=1, random_state=seed).fit(X_scaled)
    single_runs.append({
        "seed": seed,
        "inertia": round(km.inertia_),
        "ARI_vs_final": round(adjusted_rand_score(base_labels, km.labels_), 3),
    })
pd.DataFrame(single_runs)

,seed,inertia,ARI_vs_final
0,1,175665,1.000
1,2,175667,0.992
2,3,175665,0.999
3,4,175665,1.000
4,5,175665,1.000
5,6,175665,1.000
6,7,175665,1.000
7,8,175665,1.000
8,9,175667,0.992
9,10,175665,0.999
